[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/littlekg87/lecture/blob/main/2025_kmooc/notebooks/02_gu_convert.ipynb)


# 2-4차시 텍스트마이닝 실습 ② — 한문 숫자를 아라비아 숫자로 변환하기

①에서 뽑아낸 `一十萬七百九十` 같은 한문 숫자를 `100790` 처럼 **계산할 수 있는 숫자**로 바꿉니다.

**왼쪽의 ▶ 버튼을 위에서부터 차례대로 누르기만 하면 됩니다.**

> 설치할 것이 없습니다. 파이썬 기본 기능(`re`, `csv`)만 사용합니다.


## 코드 설계

| 단계 | 하는 일 |
|---|---|
| **1단계** | 한문숫자 변환 코드 입력 |
| **2단계** | csv 파일 불러오기 |
| **3단계** | 구(口) 데이터 열(문장 형태)을 한문숫자로 변환 |
| **4단계** | csv 파일로 저장 |


---
## 준비 — 실습 데이터 내려받기

① 실습에서 만든 `sejong_gu_data_sample.csv` 를 내려받습니다.


In [ ]:
# 실습 데이터를 강의 깃허브 저장소에서 곧바로 내려받습니다.
# 내 컴퓨터의 파일 경로를 적을 필요가 없습니다.

GITHUB = "https://raw.githubusercontent.com/littlekg87/lecture/main/2025_kmooc"

FILES = [
    "data/text-mining/sejong_gu_data_sample.csv",
]

import os
import urllib.request

for path in FILES:
    name = path.split('/')[-1]
    try:
        urllib.request.urlretrieve(f'{GITHUB}/{path}', name)
    except Exception as e:
        raise SystemExit(
            f'내려받기에 실패했습니다: {path}\n'
            f'  · 인터넷 연결을 확인해 주세요.\n'
            f'  · 그래도 안 되면 강의 게시판에 문의해 주세요.\n'
            f'  (원인: {e})'
        )
    print(f'내려받음: {name}  ({os.path.getsize(name):,} 바이트)')

print()
print('준비 완료! 아래 칸부터 차례로 실행하세요.')

# (선택) 깃허브 대신 내 컴퓨터의 파일을 쓰고 싶다면
# 아래 두 줄 앞의 # 을 지우고 실행한 뒤 파일을 선택하세요.
# from google.colab import files
# files.upload()


---
## 1단계 — 한문숫자 변환 코드 입력

> 이 부분은 **강사가 미리 작성해 둔 코드**입니다. 그대로 실행하시면 됩니다.

한문 숫자는 세 종류의 글자로 이루어집니다.

| 종류 | 글자 | 예 |
|---|---|---|
| 낱자 | 零 〇 一 二 三 四 五 六 七 八 九 | 三 = 3 |
| 작은 단위 | 十(10) 百(100) 千(1000) | 七百 = 700 |
| 큰 단위 | 萬(10⁴) 億(10⁸) 兆(10¹²) | 十萬 = 100000 |


In [ ]:
import re
import csv

# 1단계: 한문 숫자 매핑
digits = {
    '零': 0, '〇': 0, '一': 1, '二': 2, '三': 3, '四': 4,
    '五': 5, '六': 6, '七': 7, '八': 8, '九': 9
}

units = {
    '十': 10,
    '百': 100,
    '千': 1000
}

large_units = {
    '兆': 1000000000000,
    '億': 100000000,
    '萬': 10000
}


def parse_section(section):
    """한문 숫자 섹션을 아라비아 숫자로 변환"""
    result = 0
    num = 0
    for char in section:
        if char in digits:
            num = digits[char]
        elif char in units:
            unit = units[char]
            if num == 0:
                num = 1
            result += num * unit
            num = 0
        else:
            pass  # 알 수 없는 문자 무시
    result += num
    return result


def hanja_to_number(hanja_str):
    """전체 한문 숫자 문자열을 아라비아 숫자로 변환"""
    hanja_str = re.sub(r'[^零〇一二三四五六七八九十百千萬億兆]', '', hanja_str)
    total = 0
    parts = re.split('([兆億萬])', hanja_str)
    parts.append('')  # 마지막 단위 처리용
    i = 0
    while i < len(parts):
        section = parts[i]
        if i + 1 < len(parts) and parts[i + 1] in large_units:
            unit = large_units[parts[i + 1]]
            section_value = parse_section(section)
            total += section_value * unit
            i += 2
        else:
            total += parse_section(section)
            i += 1
    return total


print('변환 코드 준비 완료!')


### 잘 되는지 확인해 봅시다

아래 칸의 숫자를 바꿔 가며 여러 번 실행해 보세요.


In [ ]:
print(hanja_to_number('一十萬七百九十'))     # 100790
print(hanja_to_number('二萬四千一百七十'))    # 24170
print(hanja_to_number('三百'))               # 300
print(hanja_to_number('五千六百五'))          # 5605


---
## 2단계 — csv 파일 불러오기


In [ ]:
# 2단계: CSV 파일 불러오기
input_csv = 'sejong_gu_data_sample.csv'
output_csv = 'sejong_gu_data_converted.csv'

gu_data = []
with open(input_csv, mode='r', encoding='utf-8-sig') as file:
    reader = csv.reader(file)
    header = next(reader)  # 헤더 저장
    for row in reader:
        gu_data.append(row)

print(f'{len(gu_data)}개의 데이터를 불러왔습니다.')
print(gu_data[:5])


> 📌 **강의 영상과 다른 점 — 파일 경로**
>
> 영상에서는 이 자리에 이런 긴 경로가 나옵니다.
> ```python
> input_csv = r"C:\Users\littl\PycharmProjects\Sejong-Record-Practice\sejong_gu_data_sample.csv"
> ```
> 이 경로는 **강사님 컴퓨터의 주소**라서 그대로 쓰면 오류가 납니다.
> 코랩에서는 파일을 방금 내려받아 바로 옆에 두었으므로,
> **파일 이름만 적으면 됩니다.** 경로를 고칠 필요가 없습니다.


---
## 3단계 — 구(口) 데이터 열(문장 형태)을 한문숫자로 변환


In [ ]:
# 3단계: 한문 숫자 변환
converted_data = []
for row in gu_data:
    han_number = row[0]  # 첫 번째 열에 한문 숫자가 있다고 가정
    arabic_number = hanja_to_number(han_number)
    converted_data.append([han_number, arabic_number])
    print(f'  {han_number}  ->  {arabic_number}')


> 📌 **강의 자료의 주석 한 줄을 고쳤습니다.**
>
> 영상 코드에는 `row[0]` 옆에 `# 2열에 한문 숫자가 있다고 가정` 이라고 적혀 있습니다.
> 그런데 파이썬은 **0부터 세기 때문에 `row[0]` 은 2열이 아니라 1열(첫 번째 열)** 입니다.
> 코드는 원래부터 맞게 동작하고 있었고, **주석 설명만 틀렸던 것**이라 바로잡았습니다.


---
## 4단계 — csv 파일로 저장


In [ ]:
# 4단계: 변환된 데이터 다시 CSV로 저장
with open(output_csv, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(['구(口) 데이터 (한문)', '구(口) 데이터 (아라비아 숫자)'])
    for row in converted_data:
        writer.writerow(row)

print(f'한문 숫자가 변환되어 {output_csv} 파일로 저장되었습니다!')


---
## 마무리 — 만든 파일 내 컴퓨터로 내려받기


In [ ]:
from google.colab import files

files.download('sejong_gu_data_converted.csv')


---
### 정리

1. 텍스트마이닝을 위해서는 **정확한 연구 설계**가 필요하다.
2. 설계에 맞춰서 코드를 입력한다.

이제 이 숫자로 **통계 분석**과 **지도 시각화**를 할 수 있습니다.
